## Multi-threading, Multi-processing and Asyncio 

- are different ways to handle multiple pieces of work efficiently, but they achieve in different ways.

The most important distinction is:

**Multithreading:** multiple threads inside one process 
**Multiprocessing:** multiple independent Python processes
**Asyncio:** usually one thread + one event loop, switching between tasks whenever one is waiting. 

### 1. The mental model 

Suppose you own a restaurant and need to prepare 5 customer orders.

**Normal Sequential Python**

One chef handles everything:

Order 1 --------> done 
                  Order 2 ---------> done
                                     Order 3 ----------> done 

If each order takes 5 seconds: Total = 15 seconds

Now the three concurrency approaches differ in who does the work and how. 

### 2. Multithreading 

Imagine one kitchen with several chefs, but they share the same workspace, refrigerator, utensils, and ingredients.


            Python Process
       ┌──────────────────────┐
       │ Thread 1 ── Task A   │
       │ Thread 2 ── Task B   │
       │ Thread 3 ── Task C   │
       └──────────────────────┘
             Shared Memory

Threads belong to the same process, so they share memory. 

For example:

In [5]:
import threading
import time 

def download_file(file):
    print (f"Starting - {file}")
    time.sleep(2)  # simulate network request
    print (f"Finished - {file}")

files = ["a.pdf", "b.pdf", "c.pdf"]

threads = []

for idx, file in enumerate(files, start=1):
    thread = threading.Thread(target=download_file, args=(file,))
    thread.start()
    threads.append(thread)



Starting - a.pdf

Starting - b.pdf

Starting - c.pdf


Finished c.pdf
Finished a.pdf
Finished b.pdf


This works well because while one thread waits for network I/O, Python can run another.

#### Where multithreading is good
It is especially useful for I/O-bound work, such as:
- API Calls 
- Database queries 
- Downloading files
- Reading / writing files 
- Web scraping 
- Socket communication 

There is, however, an important Python-specific concept here: the GIL — Global Interpreter Lock.

In regular CPython, historically, only one thread could execute Python bytecode at a time under the GIL. That means threads were generally not the best way to speed up pure Python CPU-heavy workloads.

Conceptually:

Thread 1  ████          ████
Thread 2      ████          ████
Thread 3          ████
          ↑
     CPU execution alternates

Threads can still be extremely useful because waiting on I/O releases opportunities for other threads to run.


### 3. Multiprocessing 

Multiprocessing is fundamentally different. 

Instead of: 

One Process
|-- Thread 1 
|-- Thread 2
|-- Thread 3

You create: 

Process 1           Process 2           Process 3
Python              Python              Python
Task A              Task B              Task C
   |                  |                   |
CPU Core 1          CPU Core 2          CPU Core 3

Each process has its own Python interpreter and memory space.

Example:

In [7]:
### THIS DOES NOT WORK IN JUPYTER NOTEBOOK
from multiprocessing import Process
import time 

def calculate(n):
    print (f"Starting {n}")

    total = 0 

    for i in range(50_000_000):
        total += i

    print (f"Finished {n}")

if __name__ == "__main__":
    processes = []

    for n in range(4):
        process = Process(
            target=calculate, 
            args=(n,)
        )

        process.start()
        processes.append(process)

    for process in processes:
        process.join()


In [8]:
from multiprocessing import Process
from worker import calculate

if __name__ == "__main__":
    processes = []
    for n in range(4):
        process = Process(target=calculate, args=(n,))
        process.start()
        processes.append(process)

    for process in processes:
        process.join()


Starting 3
Starting 1
Starting 2
Starting 0
Finished 2
Finished 0
Finished 1
Finished 3



#### Option B: Use joblib (Easiest inside a single notebook)

If you want to keep everything in one notebook cell without external files, install and use joblib, which handles Jupyter serialization automatically:


In [ ]:
from joblib import Parallel, delayed

def calculate(n):
    print(f"Starting {n}")
    total = 0
    for i in range(50_000_000):
        total += i
    print(f"Finished {n}")
    return total

results = Parallel(n_jobs=4)(delayed(calculate)(n) for n in range(4))


In standard Python, if you spawn 4 threads for a CPU-heavy loop (like your calculate function), only 1 CPU core is actively executing Python code at any given instant—even if your OS scheduler bounces those threads across different cores.

This happens because of Python's GIL (Global Interpreter Lock), which makes a sharp distinction between multithreading and multiprocessing:

#### 1. Multithreading (threading module) → 1 Core at a Time

• The GIL bottleneck: Standard Python (CPython) uses a lock called the GIL that allows only one thread to execute Python bytecode at a time within a single process.

• What happens with 4 threads: Your operating system may still schedule those 4 threads across 4 different physical cores (switching which core runs the active thread every few milliseconds), so your CPU monitor might show slight activity across multiple cores. 

However, only one thread ever makes progress at a time—the other 3 sit idle waiting for the GIL.

• Best for: I/O-bound tasks (waiting on APIs, database queries, or reading files), where a thread releases the GIL while waiting.

#### 2. Multiprocessing (multiprocessing module) → 4 Cores Simultaneously

• Bypassing the GIL: Because your original script uses multiprocessing.Process, it spawns 4 separate processes, each with its own Python interpreter and its own separate GIL.

• What happens with 4 processes: Your OS scheduler assigns each process to a different CPU core, allowing all 4 calculate loops to run truly in parallel at the exact same time.

• Best for: CPU-bound tasks (heavy math, loops, data transformations).


#### How to See It Live on Your Computer

You can watch the cores light up in real time while running your 4-process script:

• On macOS: Open Activity Monitor → press Cmd + 2 (or click Window → CPU History) to see a live bar graph for every individual core.

• On Windows: Open Task Manager → Performance → CPU → right-click the graph and select Change graph to → Logical processors.

Given below, a benchmark snippet you can run to compare the exact execution time of 4 threads vs. 4 processes on your machine?


In [9]:
%pip install joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [joblib]


In [10]:
import time
from joblib import Parallel, delayed

# CPU-heavy dummy task
def calculate(n):
    total = 0
    for i in range(50_000_000):
        total += i
    return total

# 1. Baseline: Single Core (Sequential)
start = time.perf_counter()
for n in range(4):
    calculate(n)
single_time = time.perf_counter() - start
print(f"1. Sequential (1 core):      {single_time:.2f} seconds")

# 2. Multithreading: 4 Threads (Limited by Python's GIL)
start = time.perf_counter()
Parallel(n_jobs=4, prefer="threads")(delayed(calculate)(n) for n in range(4))
thread_time = time.perf_counter() - start
print(f"2. Multithreading (4 threads): {thread_time:.2f} seconds")

# 3. Multiprocessing: 4 Processes (True Parallel across 4 cores)
start = time.perf_counter()
Parallel(n_jobs=4)(delayed(calculate)(n) for n in range(4))
process_time = time.perf_counter() - start
print(f"3. Multiprocessing (4 cores):  {process_time:.2f} seconds")


1. Sequential (1 core):      2.78 seconds
2. Multithreading (4 threads): 2.96 seconds
3. Multiprocessing (4 cores):  0.89 seconds


Those numbers confirm that multithreading is restricted to one core at a time (2.96s), while multiprocessing successfully distributes the work across 4 cores simultaneously (0.89s).

Breakdown of Your Results

• Sequential (2.78s): One core runs all four loops back-to-back (~0.70s per loop).

• Multithreading (2.96s): The Global Interpreter Lock (GIL) forces all four threads to take turns on a single core. It actually runs 0.18s slower than sequential execution because the CPU wastes extra cycles constantly pausing one thread and switching to another.

• Multiprocessing (0.89s): That is a 3.1× speedup over sequential execution. Each loop (~0.70s) runs in parallel on its own core, plus roughly 0.19s of one-time overhead to spin up the 4 separate worker processes and collect the results.

Practical Rule of Thumb for Your Code

• Use multiprocessing (or joblib) when your code does heavy CPU work (pure Python loops, complex calculations, image/audio processing) and each task takes at least ~0.1 seconds—otherwise process startup overhead outweighs the speedup.

• Use threading (or asyncio) when your code spends most of its time waiting on external things (making API calls, querying databases, reading/writing files), because Python releases the GIL while waiting for I/O.

• Bonus shortcut for math/data: Libraries like NumPy, Pandas, and Polars execute their heavy math in compiled C/Rust under the hood and automatically release the GIL, giving you multi-core speed without manually managing processes.


So, multi-processing appropriate for CPU-bound work. 
Examples:
- Image processing
- Video encoding
- Large mathematical calculations
- Data transformations
- Scientific computing
- ML preprocessing
- Simulation

But processes are heavier than threads. 

Thread
   ↓
shares process memory
cheap to create
easy communication

Process
   ↓
separate memory
more expensive to create
requires IPC / serialization

So multiprocessing isn't automatically "better."

### 4. Asyncio

Asyncio is where things get especially interesting.

There may be:
1 process
1 thread
1 CPU core

yet the program can efficiently manage thousands of concurrent tasks.

The secret is the event loop.

                EVENT LOOP
                    │
        ┌───────────┼───────────┐
        ↓           ↓           ↓
      Task A      Task B      Task C
        │           │           │
      waiting     waiting     running
       HTTP          DB

Instead of creating threads, tasks voluntarily give control back to the event loop when they are waiting.

The key word is: await


In [12]:
import asyncio

async def download_file(file):
    print (f"Starting {file}")
    await asyncio.sleep(2)
    print (f"Finished {file}")

async def main():
    await asyncio.gather(
        download_file("a.pdf"),
        download_file("b.pdf"),
        download_file("c.pdf"),
        return_exceptions=False
    )

await main()

Starting a.pdf
Starting b.pdf
Starting c.pdf
Finished a.pdf
Finished b.pdf
Finished c.pdf


3 processes - Total 2 seconds

Notice that this looks similar to the multithreading example.

But internally it is very different.
In Threading, the operating system is switching between threads.

OS Scheduler

    Thread A
        ↓
    Thread B
        ↓
    Thread C

Asyncio
Your Python event loop decides which coroutine should run.

Python Event Loop

Task A
  │
  └── await network
           ↓
         Task B
           │
           └── await database
                    ↓
                  Task C

This is called cooperative concurrency.

#### 5. Why await matters

Consider: